In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
def remove_unwanted_columns(df):
    """
    Removes specific unwanted columns if they exist in the DataFrame.
    """
    unwanted = [
        'Unnamed: 0', 'nox', 'bp', 'rainfall', 'time_idx', 'Month',
        'day_of_week', 'Day_of_week', 'pm25_lag1', 'pm25_lag7', 'to_date'
    ]
    
    # Only drop columns that are actually present in the dataframe
    cols_to_drop = [col for col in unwanted if col in df.columns]
    
    return df.drop(columns=cols_to_drop)



def fix_temporal_structure(df, datetime_col='from_date', freq='H'):
    """
    Ensures the dataframe has a continuous datetime index with no gaps.
    """
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    
    # Remove duplicates by taking the mean of entries with the same timestamp
    df = df.groupby(datetime_col).mean(numeric_only=True).reset_index()

    # Set index and fill missing hours
    df = df.set_index(datetime_col)
    df = df.asfreq(freq)
    
    return df



def apply_physical_bounds(df):
    """
    Clips sensor data to realistic physical limits.
    """
    # Pollutants & Wind Speed: Cannot be negative
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].clip(lower=0)
    
    # Humidity: Cannot exceed 100%
    if 'humidity' in df.columns:
        df['humidity'] = df['humidity'].clip(upper=100)
        
    return df



def handle_missing_values(df, max_gap=3):
    """
    Fills gaps using linear interpolation. 
    'max_gap' limits interpolation so we don't 'guess' too much data.
    """
    # Interpolate small gaps linearly
    df = df.interpolate(method='linear', limit=max_gap)
    
    # For larger gaps, use a backfill/forward fill to catch edges
    df = df.ffill().bfill()
    
    return df

In [3]:
def load_and_preprocess_split(train_path, val_path, test_path):
    """
    Loads three CSVs, cleans them using the predefined pipeline, 
    and returns scaled DataFrames + the fitted scaler.
    """
    
    # 1. Load raw data
    datasets = {
        'train': pd.read_csv(train_path),
        'val': pd.read_csv(val_path),
        'test': pd.read_csv(test_path)
    }
    
    processed_dfs = {}
    
    for name, df in datasets.items():
        # Apply the pipeline we built earlier
        # Note: 'city' and 'to_date' are usually dropped for modeling
        df_clean = (df.pipe(remove_unwanted_columns)  # 1. Strip junk first
                      .pipe(fix_temporal_structure)    # 2. Fix time index
                      .pipe(apply_physical_bounds)    # 3. Clip sensor errors
                      .pipe(handle_missing_values))    # 4. Fill NaNs
        
        
        processed_dfs[name] = df_clean

    # 2. Feature Scaling (Standardization)
    # We fit ONLY on train to prevent data leakage
    scaler = StandardScaler()
    feature_cols = processed_dfs['train'].columns
    
    # Fit on train
    scaler.fit(processed_dfs['train'][feature_cols])
    
    # Transform all sets
    for name in processed_dfs:
        scaled_values = scaler.transform(processed_dfs[name][feature_cols])
        processed_dfs[name] = pd.DataFrame(
            scaled_values, 
            columns=feature_cols, 
            index=processed_dfs[name].index
        )
        
    return processed_dfs['train'], processed_dfs['val'], processed_dfs['test'], scaler

In [4]:
train_data, val_data, test_data, fitted_scaler = load_and_preprocess_split('./data/Train_data.csv', './data/Validation_data.csv', './data/Test_data.csv')

C:\Users\moreh\AppData\Local\Temp\ipykernel_9184\2380578284.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq(freq)
C:\Users\moreh\AppData\Local\Temp\ipykernel_9184\2380578284.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq(freq)
C:\Users\moreh\AppData\Local\Temp\ipykernel_9184\2380578284.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq(freq)


In [5]:
train_data.head(2)

,pm25,pm10,no,nh3,no2,so2,co,ozone,wind_speed,air_temp,humidity
from_date,,,,,,,,,,,
2017-01-01 00:00:00,1.202727,0.820378,0.466865,2.999675,0.420749,0.498072,0.888652,-0.487698,-0.895621,-1.630654,1.286546
2017-01-01 01:00:00,1.305182,0.992247,0.550120,2.958012,0.380645,0.282169,1.243119,-0.510335,-0.846712,-1.695370,1.318042


In [6]:
val_data.head(2)

,pm25,pm10,no,nh3,no2,so2,co,ozone,wind_speed,air_temp,humidity
from_date,,,,,,,,,,,
2023-01-01 00:00:00,0.667615,0.132173,0.376445,1.361848,-0.286300,-0.897499,0.173424,-1.025835,-0.972897,-1.660701,1.290660
2023-01-01 01:00:00,0.603785,0.073588,0.113052,1.329473,-0.424331,-0.990943,-0.055355,-1.001256,-1.060933,-1.687900,1.306465


In [7]:
test_data.head(2)

,pm25,pm10,no,nh3,no2,so2,co,ozone,wind_speed,air_temp,humidity
from_date,,,,,,,,,,,
2023-03-01 00:00:00,-0.379788,0.176568,1.275962,0.442174,0.139315,-0.695474,0.306829,-1.034077,-0.521630,-0.748260,-0.167973
2023-03-01 01:00:00,-0.298469,0.362027,0.871993,0.707168,-0.025709,-0.763537,0.120446,-1.000238,-0.669661,-0.728882,-0.281468


# Method 1: Predict PM2.5

# Method 2: Predict NO2

# Method 3: Predict CO

# Method 4: Predict Ozone

# Method 5: Predict All (Multi-variate/ Multi-task)